# DP2 —  Source catalog query over a DDF via Butler

**Author:** dagoret  
**Creation Date:** 2026-03-17  
**Last Date:** 2026-03-17  
**version:** v0
## Purpose

Retrieve **Object**, **Source**, and **ForcedSource** catalogs
for a user-selected Deep Drilling Field (DDF) using **Butler Gen3 only**
(no TAP service, not yet available for DP2 at USDF).

## Actual schema (discovered at runtime — COSMOS, LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545)

- `dia_object` : dims `(skymap, tract)` — 1 ref per tract, index = `diaObjectId`
- `dia_source` : dims `(skymap, tract)` — 1 ref per tract, index = `diaSourceId` (join on `diaObjectId` column)
- `dia_object_forced_source` : dims `(skymap, tract, patch)` — **21 refs per tract** (one per patch)
  - Must iterate over all patch refs and concat
  - Contains per-visit forced photometry linked to DiaObjects

## Reference notebooks
- `2026-03-07_AccessToDP2.ipynb` — Butler setup
- `2026-03-13_DP2_DDF_Tracts_SkyMap_Mollweide.ipynb` — tract lookup

---
## 0. Imports

In [ ]:
import warnings

warnings.filterwarnings("ignore")
import traceback
import gc


import os

import numpy as np
import pandas as pd
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from astropy.coordinates import SkyCoord
from astropy.table import Table
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import Angle

from scipy.stats import median_abs_deviation

import lsst
from lsst.daf.butler import Butler
from lsst.geom import SpherePoint, degrees
import lsst.geom as geom

print(f"Matplotlib : {matplotlib.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("Imports OK")

In [ ]:
mpl.rcParams.update(
    {
        "figure.figsize": (8, 5),  # default figure size
        "font.size": 14,  # taille de base
        "axes.titlesize": 18,  # titre de l'axe
        "axes.labelsize": 16,  # labels x et y
        "xtick.labelsize": 14,  # ticks x
        "ytick.labelsize": 14,  # ticks y
        "legend.fontsize": 14,  # legend text
        "legend.title_fontsize": 15,  # legend title
        "figure.titlesize": 20,  # titre global de la figure
    }
)

In [ ]:
# Remove to run faster the notebook
# import ipywidgets as widgets
# %matplotlib widget

# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
# try:
#    import ipympl  # noqa: F401
#    %matplotlib widget
#    print("ipympl found → interactive backend (%matplotlib widget)")
# except ImportError:
#    %matplotlib inline
#    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
#    print("Install with:  pip install ipympl")


In [ ]:
# output dirs
NB_TAG = "DP2_DDF_STATICOBJ_BUTLER_01"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")


# Output directory for DRP data
OUTPUT_DIR = "data_DP2_DDF_OBJECT_DRP_01"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Visits filename
FILENAME_VISITS = "visitTable-2025041500138-2026010800515_N34276.csv"

In [ ]:
# FLAGS
FLAG_FETCH_VISITSEXPOSURES = False

In [ ]:
# =========================================================================
# Helper: add a top x-axis showing calendar dates above the MJD axis
# =========================================================================
# The bottom x-axis of each light curve panel uses MJD.
# This helper adds a twin axis on top with evenly-spaced date labels
# (UTC) so you can immediately read off known observing dates.
#
# Usage (call AFTER plotting, BEFORE tight_layout / savefig):
#   add_date_top_axis(axes[0], n_ticks=6)
# =========================================================================


def add_date_top_axis(ax, n_ticks=6, date_fmt="%Y-%m-%d"):
    """
    Add a secondary x-axis on top of `ax` showing calendar dates.

    The secondary axis shares the same MJD limits as the primary axis
    but labels ticks as UTC calendar dates (e.g. 2025-06-15).

    Parameters
    ----------
    ax : matplotlib Axes
        Primary axes whose bottom x-axis carries MJD values.
    n_ticks : int
        Number of evenly-spaced tick marks on the top axis.
    date_fmt : str
        strftime format for the date labels (default YYYY-MM-DD).

    Returns
    -------
    ax_top : matplotlib Axes
        The new twin axes placed on top.
    """
    mjd_min, mjd_max = ax.get_xlim()

    # Build evenly-spaced MJD tick positions spanning the current x range
    mjd_ticks = np.linspace(mjd_min, mjd_max, n_ticks)

    # Convert each MJD tick to a calendar date string via astropy
    date_labels = [Time(m, format="mjd", scale="utc").to_value("isot")[:10] for m in mjd_ticks]

    # Create a twin axis that shares the same x-scale
    ax_top = ax.twiny()
    ax_top.set_xlim(mjd_min, mjd_max)  # must match primary axis exactly
    ax_top.set_xticks(mjd_ticks)
    ax_top.set_xticklabels(date_labels, rotation=30, ha="left", fontsize=8)
    ax_top.set_xlabel("Date (UTC)", fontsize=9, labelpad=4)

    return ax_top


print("add_date_top_axis() helper defined.")

In [ ]:
def getLostOfVisits(registry, where_clause_date):
    """
    Get exposure/visits from butler registry
    parameters:
    ==========
        registry : butler registry
        where_clause_date : specify which instrument and exposure dates in the butler registry
    """
    df_exposure = pd.DataFrame(
        {
            "id": pd.Series(dtype="int"),
            "obs_id": pd.Series(dtype="int"),
            "day_obs": pd.Series(dtype="int"),
            "seq_num": pd.Series(dtype="int"),
            "time_start": pd.Series(dtype="str"),  # ou 'datetime64[ns]' si c'est un datetime
            "time_end": pd.Series(dtype="str"),  # idem
            "type": pd.Series(dtype="str"),
            "target": pd.Series(dtype="str"),
            "filter": pd.Series(dtype="str"),
            "zenith_angle": pd.Series(dtype="float"),
            "expos": pd.Series(dtype="float"),  # ou 'int' selon le cas
            "ra": pd.Series(dtype="float"),
            "dec": pd.Series(dtype="float"),
            "skyangle": pd.Series(dtype="float"),
            "azimuth": pd.Series(dtype="float"),
            "zenith": pd.Series(dtype="float"),
            "science_program": pd.Series(dtype="str"),
            "jd": pd.Series(dtype="float"),
            "mjd": pd.Series(dtype="float"),
        }
    )

    # save the data array in rows before saving in pandas dataframe
    rows = []
    for count, info in enumerate(registry.queryDimensionRecords("exposure", where=where_clause_date)):
        try:
            jd_start = info.timespan.begin.value
            jd_end = info.timespan.end.value
            the_Time_start = Time(jd_start, format="jd", scale="utc")
            the_Time_end = Time(jd_end, format="jd", scale="utc")
            mjd_start = the_Time_start.mjd
            mjd_end = the_Time_end.mjd
            isot_start = the_Time_start.isot
            isot_end = the_Time_end.isot

            if count == 0:
                print("===== Time Conversion Debug Info =====")
                print(f"JD start      : {jd_start} (type: {type(jd_start)})")
                print(f"JD end        : {jd_end} (type: {type(jd_end)})")
                print(f"MJD start     : {mjd_start} (type: {type(mjd_start)})")
                print(f"MJD end       : {mjd_end} (type: {type(mjd_end)})")
                print(f"ISOT start    : {isot_start} (type: {type(isot_start)})")
                print(f"ISOT end      : {isot_end} (type: {type(isot_end)})")
                print("=======================================")

            # put row in a dictionnary before stacking
            row = {
                "id": info.id,
                "obs_id": info.obs_id,
                "day_obs": info.day_obs,
                "seq_num": info.seq_num,
                "time_start": isot_start,
                "time_end": isot_end,
                "type": info.observation_type,
                "target": info.target_name,
                "filter": info.physical_filter,
                "zenith_angle": info.zenith_angle,
                "expos": info.exposure_time,  # Exemple : adapter selon ton objet
                "ra": info.tracking_ra,
                "dec": info.tracking_dec,
                "skyangle": info.sky_angle,
                "azimuth": info.azimuth,
                "zenith": info.zenith_angle,
                "science_program": info.science_program,
                "jd": float(jd_start),
                "mjd": float(mjd_start),
            }
            rows.append(row)

        except ValueError as e:
            print(f"Erreur de valeur : {e}")
        except FileNotFoundError as e:
            print(f"Fichier introuvable : {e}")
        except Exception as e:
            print(f"Erreur inattendue : {type(e).__name__} - {e}")
            print(f">>>   Unexpected error at row {count}:", sys.exc_info()[0])
            traceback.print_exc()  # displays the full stack trace

    # Final DataFrame creation
    df_exposure = pd.DataFrame(rows)
    df_science = df_exposure[df_exposure.type == "science"]
    df_science.reset_index(drop=True, inplace=True)

    return df_science

In [ ]:
def load_visits_file(file_path):
    """
    Checks if the visits file exists and loads it into a DataFrame.
    """
    if not os.path.exists(file_path):
        print(f"ERROR: File not found at {file_path}")
        return None

    try:
        # Standard LSST tables are usually Parquet
        if file_path.endswith(".parquet"):
            df_visits = pd.read_parquet(file_path)
        else:
            df_visits = pd.read_csv(file_path)

        print(f"Successfully loaded {len(df_visits):,} visits from {os.path.basename(file_path)}")
        return df_visits
    except Exception as e:
        print(f"ERROR loading file: {e}")
        return None


# --- Usage ---
# df_visits = load_visits_file(FULLFILENAME_VISITS)

In [ ]:
from lsst.daf.butler import Butler
import lsst.geom as geom


def find_ddf_regions(skymap, ra_deg, dec_deg, radius_deg):
    """
    Finds all tracts and patches that overlap a circular DDF region.
    """
    # 1. Define the center and the region
    center = geom.SpherePoint(ra_deg * geom.degrees, dec_deg * geom.degrees)
    # LSST geometry uses 'ConvexPolygon' or 'Circle' for searches
    # For a simple check, we can use findTracts within a radius

    # 2. Find overlapping tracts
    # findTractsList returns tracts that intersect the circle
    tract_list = skymap.findTractsList(center, radius_deg * geom.degrees)

    results = []

    for tract_info in tract_list:
        tract_id = tract_info.getId()
        wcs = tract_info.getWcs()

        # 3. Find overlapping patches within this tract
        # We look for patches that overlap our DDF circle
        patch_list = tract_info.getSequentialPatchNumbers(center, radius_deg * geom.degrees)

        for patch_id in patch_list:
            # Convert sequential ID to (x, y) patch coordinates (e.g., '1,1')
            patch_index = tract_info.getPatchIndex(patch_id)
            patch_name = f"{patch_index[0]},{patch_index[1]}"

            results.append({"tract": tract_id, "patch": patch_name, "patch_inner_id": patch_id})

    return results


# --- Example Usage ---
# butler = Butler(REPO_PATH)
# skymap = butler.get('skyMap')

# ELAIS-S1 center example: RA=9.45, Dec=-44.0, Radius~1.75 deg
# ddf_regions = find_ddf_regions(skymap, 9.45, -44.0, 1.75)

---
## 1. User Configuration

In [ ]:
CONE_RADIUS_DEG = 1.8
MIN_NSOURCES = 500

REPO = "dp2_prep"
COLLECTION = "LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545"
SKYMAP_NAME = "lsst_cells_v2"
INSTRUMENT = "LSSTCam"
WHERE_CLAUSE_INSTRUMENT = "instrument = '" + f"{INSTRUMENT}" + "'"
DATE_START = 20250415
WHERE_CLAUSE_DATE = WHERE_CLAUSE_INSTRUMENT + f" and day_obs >= {DATE_START}"

TRACT_SEARCH_RADIUS_DEG = 1.8
DDF_RADIUS = TRACT_SEARCH_RADIUS_DEG

DDF_COORDS = {
    "COSMOS": (150.119, +2.206),
    "ECDFS": (53.125, -28.100),
    "ELAIS-S1": (9.450, -44.000),
    "XMM-LSS": (35.708, -4.750),
    "EDFS-a": (58.900, -49.315),
    "EDFS-b": (63.600, -47.600),
    "EDFS": (61.240, -48.423),
    "M49": (187.400, +8.000),
}


print(f"Cone radius : {CONE_RADIUS_DEG}°")
print(f"Min nDiaSrc : {MIN_NSOURCES}")
print(f"Collection  : {COLLECTION}")
print(f"Instrument  : {INSTRUMENT}")
print(f"Where clause  : {WHERE_CLAUSE_INSTRUMENT}")
print(f"Where clause date : {WHERE_CLAUSE_DATE}")

---
## 2. Butler and SkyMap

In [ ]:
butler = Butler(REPO, collections=COLLECTION)
registry = butler.registry
print("Butler OK")

In [ ]:
try:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME, collections=COLLECTION)
except Exception:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME)
print(f"SkyMap '{SKYMAP_NAME}' : {len(skymap)} tracts")

### 2.2 Retrieve the science visits in order to match the visitid with the MJD in forced source photometry
- `visitid = id`
- `MJD = mjd `

In [ ]:
if FLAG_FETCH_VISITSEXPOSURES:
    visitTable = getLostOfVisits(registry, where_clause_date=WHERE_CLAUSE_DATE)
    print(visitTable.head(1))
    visitMin, visitMax, Nvisits = visitTable["id"].min(), visitTable["id"].max(), len(visitTable)
    print(f"visitMin = {visitMin},visitMax = {visitMax}, Nvisits = {Nvisits}")
    visit_filename = f"visitTable-{visitMin}-{visitMax}_N{Nvisits}.csv"
    print(f"filename_visits = {visit_filename}")
    visit_fullfilename = os.path.join(OUTPUT_DIR, visit_filename)
    visitTable.to_csv(visit_fullfilename)
    del visitTable
    gc.collect()

---
## 3. Dataset Type Inventory

Identify the correct names and **dimensions** for DIA products — in particular
`dia_object_forced_source` has `(skymap, tract, patch)` dimensions, which
requires iterating over patch refs.

In [ ]:
# Hard-coded after discovery: confirmed present in DM-54249
DATASET_TYPE_OBJ = "object"  # dims: skymap, tract
DATASET_TYPE_SRC = "source"  # dims: skymap, tract
DATASET_TYPE_FORCED = "object_forced_source"  # dims: skymap, tract, patch


# Print actual dimensions for confirmation
for dt_name in [DATASET_TYPE_OBJ, DATASET_TYPE_SRC, DATASET_TYPE_FORCED]:
    try:
        dt = registry.getDatasetType(dt_name)
        in_col = registry.queryDatasets(dt_name, collections=COLLECTION).any(execute=False, exact=False)
        print(f"{dt_name:40s}  dims={list(dt.dimensions.names)}  present={in_col}")
    except Exception as exc:
        print(f"{dt_name:40s}  ERROR: {exc}")

---
## 4. Find Tracts Covering the DDF

In [ ]:
def find_tracts_for_coord(skymap, ra_deg, dec_deg, radius_deg=1.8):
    cos_dec = max(np.cos(np.deg2rad(dec_deg)), 0.01)
    step = 0.35
    found_ids = set()
    for ddec in np.arange(-radius_deg, radius_deg + step, step):
        for dra in np.arange(-radius_deg, radius_deg + step, step):
            if np.sqrt(dra**2 + ddec**2) > radius_deg:
                continue
            ra_s = ra_deg + dra / cos_dec
            dec_s = dec_deg + ddec
            if not (-89.9 <= dec_s <= 89.9):
                continue
            sp = SpherePoint(ra_s * degrees, dec_s * degrees)
            try:
                found_ids.add(skymap.findTract(sp).tract_id)
            except Exception:
                pass
    return sorted(found_ids)


# tract_ids = find_tracts_for_coord(skymap, RA_CENTER, DEC_CENTER, radius_deg=TRACT_SEARCH_RADIUS_DEG)
# print(f"Tracts covering {SELECTED_DDF}: {tract_ids}")

In [ ]:
DDF_COORDS = {
    "COSMOS": (150.119, +2.206),
    "ECDFS": (53.125, -28.100),
    "ELAIS-S1": (9.450, -44.000),
    "XMM-LSS": (35.708, -4.750),
    "EDFS-a": (58.900, -49.315),
    "EDFS-b": (63.600, -47.600),
    "EDFS": (61.240, -48.423),
    "M49": (187.400, +8.000),
}

for ddf_item in DDF_COORDS.items():
    ddf_name = ddf_item[0]
    RA_CENTER = ddf_item[1][0]
    DEC_CENTER = ddf_item[1][1]
    tract_ids = find_tracts_for_coord(skymap, RA_CENTER, DEC_CENTER, radius_deg=TRACT_SEARCH_RADIUS_DEG)
    print(f"{ddf_name} :: ", tract_ids)

## 5. Geometrical Mapping

In [ ]:
ddf_geometry_mapping = []

for name, (ra, dec) in DDF_COORDS.items():
    center = geom.SpherePoint(ra * geom.degrees, dec * geom.degrees)
    r_ang = DDF_RADIUS * geom.degrees

    # 1. We retrieve ALL tracts for this DDF (your find_tracts_for_coord function)
    # ou via skymap.findTractsList([points_de_la_circonference])
    tract_ids = find_tracts_for_coord(skymap, ra, dec, radius_deg=DDF_RADIUS)

    # Points de test pour couvrir la surface du DDF dans chaque tract
    coord_list = [
        center,
        center.offset(r_ang, 0 * geom.degrees),  # Nord
        center.offset(-r_ang, 0 * geom.degrees),  # Sud
        center.offset(0 * geom.degrees, r_ang),  # Est
        center.offset(0 * geom.degrees, -r_ang),  # Ouest
    ]

    for t_id in tract_ids:
        tract_info = skymap.generateTract(t_id)  # We generate the TractInfo object

        # 2. We look for which patches of THIS specific tract are in the DDF
        try:
            # findPatchList renvoie les patches qui contiennent ces points
            patch_info_list = tract_info.findPatchList(coord_list)

            for patch_info in patch_info_list:
                idx = patch_info.getIndex()
                ix = idx[0]
                iy = idx[1]
                patch_name = f"{ix},{iy}"
                n_patches_x = tract_info.getNumPatches()[0]
                n_patches_y = tract_info.getNumPatches()[1]

                seq_id = iy * n_patches_x + ix

                ddf_geometry_mapping.append(
                    {"ddf_name": name, "tract": t_id, "patch": patch_name, "patch_id": seq_id}
                )

        except (lsst.pex.exceptions.LsstCppException, IndexError, TypeError) as e:
            # This catches the "Bug" you feared (projection errors)
            # and prevents the kernel from dying.
            print(f"Skipping Tract {t_id} due to geometry boundary: {str(e)[:50]}...")
            continue

        # Catching the broad range of LSST and Python errors that occur at boundaries
        except (RuntimeError, IndexError, TypeError, Exception) as e:
            # We print just the first bit of the error to keep logs clean
            print(f"Skipping Tract {t_id}: Boundary or Projection issue.")
            continue

        except Exception as e:
            # General catch-all for anything else
            print(f"Unexpected error in Tract {t_id}: {e}")
            continue

In [ ]:
df_ddf_map = pd.DataFrame(ddf_geometry_mapping).drop_duplicates()
print(f"Mapping done : {len(df_ddf_map)} combinaisons Tract/Patch found.")

---
## 5. Schema Discovery (one-time probe)

### 5.1 Object Schema

In [ ]:
refs_probe = list(
    registry.queryDatasets(
        DATASET_TYPE_OBJ,
        collections=COLLECTION,
        dataId={"skymap": SKYMAP_NAME, "tract": tract_ids[0]},
    )
)
assert refs_probe, "No  object ref found."

obj_raw = butler.get(refs_probe[0])
df_probe = obj_raw.to_pandas() if hasattr(obj_raw, "to_pandas") else obj_raw

print(f"index.name  : {df_probe.index.name!r}")
print(f"index.dtype : {df_probe.index.dtype}")
print(f"index[:3]   : {df_probe.index[:3].tolist()}")
# print(f"columns     : {list(df_probe.columns)}")

In [ ]:
# Determine ID column
if df_probe.index.name is not None:
    OBJ_ID_COL = df_probe.index.name
    ID_IN_INDEX = True
    print(f"Object ID in index: '{OBJ_ID_COL}'")
elif any(c in df_probe.columns for c in ["objectId", "object_id"]):
    OBJ_ID_COL = next(c for c in ["objectId", "object_id"] if c in df_probe.columns)
    ID_IN_INDEX = False
    print(f"Object ID as column: '{OBJ_ID_COL}'")
else:
    OBJ_ID_COL = "row_id"
    ID_IN_INDEX = False
    print("WARNING: no ID found, using row index.")

In [ ]:
# Also probe object_forced_source schema (patch-level)
refs_forced_probe = list(
    registry.queryDatasets(
        DATASET_TYPE_FORCED,
        collections=COLLECTION,
        dataId={"skymap": SKYMAP_NAME, "tract": tract_ids[0]},
    )
)
print(f"object_forced_source refs for tract {tract_ids[0]}: {len(refs_forced_probe)}")

if refs_forced_probe:
    frc_raw = butler.get(refs_forced_probe[0])
    df_fprobe = frc_raw.to_pandas() if hasattr(frc_raw, "to_pandas") else frc_raw
    print(f"index.name  : {df_fprobe.index.name!r}")
    print(f"columns     : {list(df_fprobe.columns)}")
    print(f"shape       : {df_fprobe.shape}")

---
## 6. Helper: `to_dataframe()`

In [ ]:
def to_dataframe(obj, id_col=None, id_in_index=None):
    """
    Convert Butler catalog to pandas DataFrame.
    If id_in_index is True, promotes the named index to a regular column.
    Falls back to global OBJ_ID_COL / ID_IN_INDEX if not provided.
    """
    _id_col = id_col if id_col is not None else OBJ_ID_COL
    _id_idx = id_in_index if id_in_index is not None else ID_IN_INDEX

    if isinstance(obj, pd.DataFrame):
        df = obj
    elif hasattr(obj, "to_pandas"):
        df = obj.to_pandas()
    elif hasattr(obj, "to_table"):
        df = obj.to_table().to_pandas()
    elif isinstance(obj, Table):
        df = obj.to_pandas()
    else:
        raise TypeError(f"Cannot convert {type(obj)} to DataFrame.")

    if _id_idx and _id_col not in df.columns:
        df = df.copy()
        df.insert(0, _id_col, df.index)
        df = df.reset_index(drop=True)
    elif _id_col == "row_id" and _id_col not in df.columns:
        df.insert(0, _id_col, range(len(df)))

    return df

### 11.4 Load  Visit 

In [ ]:
FULLFILENAME_VISITS = os.path.join(OUTPUT_DIR, FILENAME_VISITS)
print(f"Visit Filename : {FULLFILENAME_VISITS} ")

In [ ]:
df_visitTable = load_visits_file(FULLFILENAME_VISITS)

In [ ]:
# Create the 'band' column by taking the first character of 'filter'
df_visitTable["band"] = df_visitTable["filter"].str[0]

# Verification: check the unique values
print("Unique bands identified:", df_visitTable["band"].unique())

# Optional: preview the result
display(df_visitTable[["target", "filter", "band"]].head())

In [ ]:
display(df_visitTable.head())

In [ ]:
df_visitTable.head()

In [ ]:
df_visitTable.target.unique()

In [ ]:
# --- 1. Define the DDF filter ---
# We look for 'y' band and any target containing 'DDF' or 'COSMOS'
is_y_band = df_visitTable["band"] == "y"
is_ddf = df_visitTable["target"].str.contains("DDF|COSMOS|ELAIS|XMM|ECDFS|EDFS", case=False, na=False)

df_y_ddf_visits = df_visitTable[is_y_band & is_ddf].copy()

# --- 2. Identify the unique nights ---
# 'dayObs' is typically the observation date (YYYYMMDD)
nights_y_ddf = sorted(df_y_ddf_visits["day_obs"].unique())

print(f"Found {len(df_y_ddf_visits)} visits in y-band for DDFs.")
print(f"Total nights involved: {len(nights_y_ddf)}")
print(f"First 5 nights: {nights_y_ddf[:]}")

In [ ]:
# 1. Filter for y-band and DDF targets
is_y = df_visitTable["band"] == "y"
is_ddf = df_visitTable["target"].str.contains("DDF|COSMOS|ELAIS|XMM|ECDFS|EDFS", case=False, na=False)

df_y_ddf = df_visitTable[is_y & is_ddf].copy()

In [ ]:
# Rayon de recherche (FoV de Rubin + marge pour inclure les DDF)
# 2.0 or 2.1 degrees is a good value to capture overlaps
SEARCH_RADIUS = 2.0

ddf_visits_list = []

for ddf_name, (ddf_ra, ddf_dec) in DDF_COORDS.items():
    ddf_center = geom.SpherePoint(ddf_ra * geom.degrees, ddf_dec * geom.degrees)

    # We iterate over the visit table
    for _, visit in df_visitTable.iterrows():
        visit_center = geom.SpherePoint(visit["ra"] * geom.degrees, visit["dec"] * geom.degrees)

        # Calcul de la distance angulaire
        dist = ddf_center.separation(visit_center).asDegrees()

        if dist <= SEARCH_RADIUS:
            ddf_visits_list.append(
                {
                    "visit_id": visit["id"],
                    "ddf_name": ddf_name,
                    "distance_deg": dist,
                    "filter": visit["filter"],
                    "mjd": visit["mjd"],
                }
            )

df_ddf_visits = pd.DataFrame(ddf_visits_list)

print(f"Visites identifiées par DDF :")
print(df_ddf_visits["ddf_name"].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from astropy.time import Time
import numpy as np

In [ ]:
# 1. We define the order of bands and associated colors
band_order = ["u", "g", "r", "i", "z", "y"]
band_colors = {"u": "purple", "g": "blue", "r": "green", "i": "orange", "z": "red", "y": "brown"}

# 2. We prepare the figure: one subplot per DDF
fig, axes = plt.subplots(len(DDF_COORDS), 1, figsize=(12, 4 * len(DDF_COORDS)), sharex=True)
if len(DDF_COORDS) == 1:
    axes = [axes]

for ax, ddf_name in zip(axes, DDF_COORDS.keys()):
    # Filtering visits for this DDF (from previous step)
    df_this_ddf = df_ddf_visits[df_ddf_visits["ddf_name"] == ddf_name].copy()

    # On arrondit le MJD pour grouper par nuit (MJD entier = midi à midi)
    df_this_ddf["night"] = np.floor(df_this_ddf["mjd"]).astype(int)

    # On boucle sur les bandes pour tracer les points
    for band in band_order:
        df_band = df_this_ddf[df_this_ddf["filter"].str.startswith(band)]

        if not df_band.empty:
            # On compte le nombre de visites par nuit
            nightly_counts = df_band.groupby("night").size()
            nights_mjd = nightly_counts.index
            counts = nightly_counts.values

            # Conversion MJD -> Datetime pour l'affichage
            dates = Time(nights_mjd, format="mjd").to_datetime()

            ax.step(dates, counts, where="mid", label=f"band {band}", color=band_colors[band], alpha=0.7)
            ax.scatter(dates, counts, color=band_colors[band], s=10)

    ax.set_ylabel("Nb Visites / Nuit")
    ax.set_title(f"DDF: {ddf_name}")
    ax.legend(loc="upper right", ncol=3, fontsize="small")
    ax.grid(True, alpha=0.3)

# 3. Formatting the X axis (Shared)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
plt.xticks(rotation=45)

plt.xlabel("Date (UTC)")
plt.tight_layout()
plt.show()

In [ ]:
# 1. Configuration
band_order = ["u", "g", "r", "i", "z", "y"]


band_colors = {
    "u": "#8000ff",  # Violet
    "g": "#228b22",  # Vert (Forest Green)
    "r": "#ff0000",  # Rouge
    "i": "#ff8c00",  # Orange (Dark Orange)
    "z": "#708090",  # Gris (Slate Gray)
    "y": "#000000",  # Noir
}


fig, axes = plt.subplots(
    len(DDF_COORDS),
    1,
    figsize=(16, 3.5 * len(DDF_COORDS)),
    sharex=True,
    gridspec_kw={"hspace": 0.05},
    constrained_layout=True,
)

if len(DDF_COORDS) == 1:
    axes = [axes]

# Limites globales pour l'axe X
mjd_min = np.floor(df_ddf_visits["mjd"].min()) - 2
mjd_max = np.ceil(df_ddf_visits["mjd"].max()) + 2

for ax, ddf_name in zip(axes, DDF_COORDS.keys()):
    # Filtrage
    df_plot = df_ddf_visits[df_ddf_visits["ddf_name"] == ddf_name].copy()

    if df_plot.empty:
        ax.set_title(f"DDF: {ddf_name} - Total: 0 visites")
        continue

    # Data preparation
    df_plot["band_clean"] = df_plot["filter"].str[0]
    df_plot["night"] = np.floor(df_plot["mjd"]).astype(int)

    # Calcul du total global pour le titre
    total_visites = len(df_plot)

    # Pivot pour le stacking
    occ_map = df_plot.groupby(["night", "band_clean"]).size().unstack(fill_value=0)
    for b in band_order:
        if b not in occ_map.columns:
            occ_map[b] = 0
    occ_map = occ_map[band_order]

    # Plotting stacked bars
    bottom = np.zeros(len(occ_map))
    for band in band_order:
        sum_band = occ_map[band].sum()  # Somme par bande
        label_str = f"Band {band} (n={sum_band})"

        ax.bar(
            occ_map.index, occ_map[band], bottom=bottom, label=label_str, color=band_colors[band], width=0.8
        )
        bottom += occ_map[band].values

    # --- CONFIGURATION DES AXES ---
    ax.set_title(f"DDF: {ddf_name} | Total Visits: {total_visites}", fontsize=14, fontweight="bold", pad=25)
    ax.set_ylabel("Visits / Night", fontsize=11)
    ax.set_xlim(mjd_min, mjd_max)
    # ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.grid(linestyle="--", alpha=0.5)

    # Axe du haut (Dates)
    ax_top = ax.secondary_xaxis("top")
    num_ticks = 10
    tick_mjds = np.linspace(mjd_min, mjd_max, num_ticks)
    ax_top.set_xticks(tick_mjds)
    ax_top.set_xticklabels(
        [Time(t, format="mjd").to_datetime().strftime("%Y-%m-%d") for t in tick_mjds],
        rotation=30,
        ha="left",
        fontsize=9,
    )

    # Legend specific to each subplot to include counts per band
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.01, 1),
        title=f"DDF: {ddf_name} ({total_visites})",
        borderaxespad=0,
    )

axes[-1].set_xlabel("MJD (Modified Julian Date)", fontsize=12)

plt.show()

In [ ]:
# 1. Configuration des styles
band_order = ["u", "g", "r", "i", "z", "y"]


band_colors = {
    "u": "#8000ff",  # Violet
    "g": "#228b22",  # Vert (Forest Green)
    "r": "#ff0000",  # Rouge
    "i": "#ff8c00",  # Orange (Dark Orange)
    "z": "#708090",  # Gris (Slate Gray)
    "y": "#000000",  # Noir
}


# 2. Definition of global time limits
tmin = np.floor(df_ddf_visits["mjd"].min()) - 5
tmax = np.ceil(df_ddf_visits["mjd"].max()) + 5

fig, axes = plt.subplots(
    len(DDF_COORDS),
    1,
    figsize=(15, 4.0 * len(DDF_COORDS)),
    sharex=False,
    gridspec_kw={"hspace": 0.05},
    constrained_layout=True,
)

if len(DDF_COORDS) == 1:
    axes = [axes]

for ax, ddf_name in zip(axes, DDF_COORDS.keys()):
    # Data filtering
    df_plot = df_ddf_visits[df_ddf_visits["ddf_name"] == ddf_name].copy()

    if df_plot.empty:
        ax.set_title(f"DDF: {ddf_name} | Total: 0 visits", fontweight="bold")
        ax.set_xlim(tmin, tmax)
        continue

    # Préparation
    df_plot["band_clean"] = df_plot["filter"].str[0]
    df_plot["night"] = np.floor(df_plot["mjd"]).astype(int)
    total_visites = len(df_plot)

    # Pivotage
    occ_map = df_plot.groupby(["night", "band_clean"]).size().unstack(fill_value=0)
    for b in band_order:
        if b not in occ_map.columns:
            occ_map[b] = 0
    occ_map = occ_map[band_order]

    # Plotting stacked bars
    bottom = np.zeros(len(occ_map))
    for band in band_order:
        sum_band = occ_map[band].sum()
        label_str = f"Bande {band} (n={sum_band})"
        ax.bar(
            occ_map.index, occ_map[band], bottom=bottom, label=label_str, color=band_colors[band], width=0.8
        )
        bottom += occ_map[band].values

    # --- AXES ET LABELS ---
    ax.set_xlim(tmin, tmax)
    ax.set_ylabel("Visits / Night", fontsize=11)
    ax.set_title(
        f"DDF: {ddf_name} | Total Visits: {total_visites}", fontsize=14, fontweight="bold", pad=35
    )  # 'pad' augmenté pour laisser de la place à la date
    # ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.grid(linestyle="--", alpha=0.5)

    # AXE SECONDAIRE (HAUT) : Dates YYYY-MM-DD
    # We use an identity function because the physical unit remains MJD
    ax_top = ax.secondary_xaxis("top", functions=(lambda x: x, lambda x: x))

    # We define 8 to 10 ticks over the [tmin, tmax] range
    tick_locations = np.linspace(tmin, tmax, 10)
    ax_top.set_xticks(tick_locations)

    # Conversion of MJDs to Date strings
    date_labels = [Time(t, format="mjd").to_datetime().strftime("%Y-%m-%d") for t in tick_locations]
    ax_top.set_xticklabels(date_labels, rotation=30, ha="left", fontsize=9, color="navy")

    # Legend to the right of each subplot
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.01, 1),
        title=f"DDF: {ddf_name} ({total_visites})",
        borderaxespad=0,
    )

# AXE PRINCIPAL (BAS) : MJD sur le dernier subplot
axes[-1].set_xlabel("MJD (Modified Julian Date)", fontsize=12, fontweight="bold", labelpad=10)

# plt.subplots_adjust(
#    left=0.1,   # Marge gauche
#    right=0.9,  # Marge droite
#    bottom=0.1, # Marge bas
#    top=0.1,    # Marge haut
#    wspace=0.4, # Espacement horizontal entre les subplots (largeur)
#    hspace=1.6  # Espacement vertical entre les subplots (hauteur)
# )

plt.show()